# Step 1: Data Cleaning and Feature Matrix Construction

This notebook covers the authoritative v2 Step 1 pipeline for the GAITNDD dataset.

**What Step 1 does:**
Loads the 64 raw `.ts` files from PhysioNet GAITNDD v1.0.0, removes artifact strides via two filters, engineers the v2 feature set, and writes the versioned feature matrix and shared control partition.

**Two artifact filters applied (in order):**
1. Stride interval > 3.0 s: removes pause events and total sensor failures (276 rows, including all 238 rows from hunt20 whose right-foot sensor was frozen throughout the recording).
2. double_support_pct > 100: removes rows with physically impossible percentage values caused by partial sensor malfunction (131 additional rows, primarily in als5 and als7).

**V2 engineered additions:**
- `asymmetry_index`: per-stride bilateral asymmetry of stride timing.
- `cv_stride`: per-subject coefficient of variation of left stride interval.
- `stride_asymmetry_signed`: signed per-stride asymmetry preserving laterality.
- `cv_swing`: per-subject coefficient of variation of left swing interval.
- `dfa_alpha_stride`: per-subject DFA exponent of the left stride-time sequence.

**Outputs written to `data/processed/`:**
- `v2/gait_features_v2.csv`: 14,753 rows x 20 columns (17 features + subject_id + condition + label)
- `control_partition.json`: fixed 8/8 control group split (Group A: training pools, Group B: cross-condition test sets)

All logic lives in `src/preprocessing.py` and `src/features.py`. This notebook imports from those modules and displays results only.


In [ ]:
import sys
import os
from pathlib import Path

sys.path.insert(0, str(Path('..') / 'src'))

import polars as pl
from features import build_feature_matrix, ALL_FEATURE_COLS

DATA_DIR = '../data/raw/gait-in-neurodegenerative-disease-database-1.0.0'
PROCESSED_DIR = '../data/processed'
OUTPUT_FILENAME = 'v2/gait_features_v2.csv'


In [ ]:
# Run the full Step 1 pipeline.
# Loads raw data, applies both artifact filters, engineers the v2 feature set,
# saves v2/gait_features_v2.csv and control_partition.json, and returns the DataFrame.
df, part = build_feature_matrix(DATA_DIR, PROCESSED_DIR, output_filename=OUTPUT_FILENAME)

print(f'Feature matrix shape: {df.shape}')
print(f'Columns ({len(df.columns)}): {df.columns}')


Feature matrix shape: (14753, 20)
Columns (20): ['left_stride_s', 'right_stride_s', 'left_swing_s', 'right_swing_s', 'left_swing_pct', 'right_swing_pct', 'left_stance_s', 'right_stance_s', 'left_stance_pct', 'right_stance_pct', 'double_support_s', 'double_support_pct', 'asymmetry_index', 'cv_stride', 'stride_asymmetry_signed', 'cv_swing', 'dfa_alpha_stride', 'subject_id', 'condition', 'label']


In [ ]:
counts = (
    df
    .group_by('condition')
    .agg(
        pl.n_unique('subject_id').alias('subjects'),
        pl.len().alias('strides'),
    )
    .sort('condition')
)
print('Subject and stride counts per condition:')
print(counts)

Subject and stride counts per condition:
shape: (4, 3)
┌───────────┬──────────┬─────────┐
│ condition ┆ subjects ┆ strides │
│ ---       ┆ ---      ┆ ---     │
│ str       ┆ u32      ┆ u32     │
╞═══════════╪══════════╪═════════╡
│ als       ┆ 13       ┆ 2404    │
│ control   ┆ 16       ┆ 4076    │
│ hd        ┆ 19       ┆ 4598    │
│ pd        ┆ 15       ┆ 3675    │
└───────────┴──────────┴─────────┘


In [ ]:
ai = df['asymmetry_index']
print('asymmetry_index (per-stride):')
print(f'  min  = {ai.min():.6f}')
print(f'  max  = {ai.max():.6f}')
print(f'  mean = {ai.mean():.6f}')

asymmetry_index (per-stride):
  min  = 0.000000
  max  = 0.928586
  mean = 0.037220


In [ ]:
# cv_stride is a subject-level property broadcast to all stride rows,
# so deduplicate by subject before computing statistics.
cv_per_subject = (
    df
    .unique(subset=['subject_id'])
    .select(['subject_id', 'cv_stride'])
    .sort('subject_id')
)
cv_vals = cv_per_subject['cv_stride']
print(f'cv_stride (per-subject, n={len(cv_per_subject)}):')
print(f'  min  = {cv_vals.min():.6f}')
print(f'  max  = {cv_vals.max():.6f}')
print(f'  mean = {cv_vals.mean():.6f}')

cv_stride (per-subject, n=63):
  min  = 0.018443
  max  = 0.239631
  mean = 0.075641


In [ ]:
print('First 5 rows:')
print(df.head(5))

First 5 rows:
shape: (5, 20)
┌────────────┬────────────┬────────────┬───────────┬───┬───────────┬───────────┬───────────┬───────┐
│ left_strid ┆ right_stri ┆ left_swing ┆ right_swi ┆ … ┆ dfa_alpha ┆ subject_i ┆ condition ┆ label │
│ e_s        ┆ de_s       ┆ _s         ┆ ng_s      ┆   ┆ _stride   ┆ d         ┆ ---       ┆ ---   │
│ ---        ┆ ---        ┆ ---        ┆ ---       ┆   ┆ ---       ┆ ---       ┆ str       ┆ i8    │
│ f64        ┆ f64        ┆ f64        ┆ f64       ┆   ┆ f64       ┆ str       ┆           ┆       │
╞════════════╪════════════╪════════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════╡
│ 1.2833     ┆ 1.3533     ┆ 0.4067     ┆ 0.4133    ┆ … ┆ 0.930972  ┆ als1      ┆ als       ┆ 1     │
│ 1.3233     ┆ 1.2667     ┆ 0.4833     ┆ 0.4       ┆ … ┆ 0.930972  ┆ als1      ┆ als       ┆ 1     │
│ 1.3033     ┆ 1.36       ┆ 0.45       ┆ 0.4267    ┆ … ┆ 0.930972  ┆ als1      ┆ als       ┆ 1     │
│ 1.4167     ┆ 1.2833     ┆ 0.5033     ┆ 0.3667    ┆ … ┆ 0.930

In [ ]:
csv_path = Path(PROCESSED_DIR) / 'v2' / 'gait_features_v2.csv'
json_path = Path(PROCESSED_DIR) / 'control_partition.json'

print(f'v2/gait_features_v2.csv exists: {csv_path.exists()}')
print(f'control_partition.json exists: {json_path.exists()}')

# Reload CSV and confirm shape matches the in-memory DataFrame.
reloaded = pl.read_csv(str(csv_path))
print(f'Reloaded CSV shape: {reloaded.shape}  (matches in-memory: {reloaded.shape == df.shape})')

# Confirm no nulls in any feature column.
null_total = df.select(ALL_FEATURE_COLS).null_count().to_numpy().sum()
print(f'Null values in feature columns: {null_total}')


v2/gait_features_v2.csv exists: True
control_partition.json exists: True
Reloaded CSV shape: (14753, 20)  (matches in-memory: True)
Null values in feature columns: 0


## Step 1 Verification Summary

| Checkpoint | Expected | Status |
|---|---|---|
| Raw strides | 15,160 | verified via wc -l on raw files |
| Removed by stride filter (> 3.0 s) | 276 | confirmed |
| Removed by pct filter (> 100%) | 131 | confirmed |
| Clean strides | 14,753 | confirmed |
| PD subjects | 15 | confirmed |
| HD subjects (usable) | 19 (hunt20 removed) | confirmed |
| ALS subjects | 13 | confirmed |
| Control subjects | 16 | confirmed |
| Feature columns | 17 | confirmed |
| Null values in feature columns | 0 | confirmed |
| v2/gait_features_v2.csv written | True | confirmed |
| control_partition.json written | True | confirmed |

Step 1 is complete. `data/processed/v2/gait_features_v2.csv` is the coordination artifact for the authoritative v2 rerun chain.
